In [15]:
import sys
import subprocess
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# sys.path.append("/home/dante/Documents/opendc/graph-greenifier-github")
sys.path.append("../")


from plottingTools import utils

# Graph Greenifier

Predicting the performance and sustainability of a data center using simulation

This demo shows Greenifier function in the Graph Massivizer project

Greenifier can also be used as a separate tool

# 1. Topology

Topologies define the available hardware of a datacenter

Defined using JSON format

[Example Topology](topologies/greenifier_topology.json)

# 2. Workloads

Workloads define what tasks need to be simulated and when

The workload used by Greenifier is similar to the output generated by Graph Optimizer

[Example workload](workloadTraces/greenifier/workload.json)

# 3. Carbon Intensity

To calculate the carbon emissions, we need information about the available energy mix

Greenifier gathers this inforamtion from the ENTSO-E project

In [11]:
df_carbon = pd.read_parquet("carbonTraces/carbon_2022.parquet")
df_carbon.head()

,timestamp,carbon_intensity
0,2021-12-31 23:00:00,168.138693
1,2021-12-31 23:15:00,167.050014
2,2021-12-31 23:30:00,164.552936
3,2021-12-31 23:45:00,167.493769
4,2022-01-01 00:00:00,164.517793


# 4. Scenario

Scenarios define what the Greenifier should simulate, and how

Scenarios are defined using a JSON format

[Example scenario](scenarios/greenifier_scenario.json)

# 5. Running Greenifier

Graph Greenifier can be run directly, using a terminal

In [16]:
subprocess.run(["../bin/Greenifier", "--scenario-path", "scenarios/greenifier_scenario.json"])



 Running scenario: 0 


Simulating... 100% [=================================] 1/1 (0:00:01 / 0:00:00) 
Simulating...   0% [                                       ] 0/1 (0:00:00 / ?) 



 Running scenario: 1 


Simulating... 100% [=================================] 1/1 (0:00:00 / 0:00:00) 


CompletedProcess(args=['../bin/Greenifier', '--scenario-path', 'scenarios/greenifier_scenario.json'], returncode=0)

## 6. Output

In [17]:
pathToOutput = "output/greenifier"

df_host = pd.read_parquet(f"{pathToOutput}/raw-output/0/seed=0/host.parquet")
df_server = pd.read_parquet(f"{pathToOutput}/raw-output/0/seed=0/server.parquet")
df_service = pd.read_parquet(f"{pathToOutput}/raw-output/0/seed=0/service.parquet")

## 7. Aggregated results

To properly compare the different experiments, we would like to aggregate them into meaningful values.

### Performance

In [18]:
runtime = utils.getTotalRuntime(df_service) 
utilization = utils.getMeanUtilization(df_host)


print(f"The total runtime of the workload was {runtime}")
print(f"On average, the utilization of each host is {utilization * 100:.2f}%")

The total runtime of the workload was 0 days 18:00:12.251000
On average, the utilization of each host is 8.28%


### Sustianability

In [19]:
energy_usage = utils.getTotalEnergyUsage(df_host, "kWh")
carbon_emissions = (df_host["carbon_emission"].sum() / 1000).round(2)

print(f"The data center used {energy_usage:.2f} kWh while running the workload")
print(f"The data center emitted {carbon_emissions:.2f} kg of carbon during the workload")

The data center used 14.40 kWh while running the workload
The data center emitted 3.44 kg of carbon during the workload


# 10. Graph Massivizer export

Greenifier provides the Graph-Choreographer with performance metrics.

These metrics are exported as a JSON file

In [20]:
output = utils.get_output(df_host, df_service, df_server, \
    save=True, exportName="output/greenifier_output.json")

[Exported File](output/greenifier_output.json)